# Minnie 1822 CAVE query

Nucleus positions, nucleus volume, proofreading status, and cell types from `minnie65_phase3_v1` at materialization 1822.

In [ ]:
from pathlib import Path

import standard_transform
from caveclient import CAVEclient

In [ ]:
CAVEclient.setup_token("https://global.daf-apis.com") # for minnie

In [ ]:
DATASTACK = "minnie65_phase3_v1"
VERSION = 1822
OUTPUT_DIR = Path("/workspaces/ConnectsCommonConnectivity/data/EM minnie 1822")

client = CAVEclient(DATASTACK, version=VERSION)
print(client.materialize.get_version_metadata(VERSION)["time_stamp"])

## nucleus positions

### cave query

In [ ]:
nuc_df = client.materialize.query_table(
    "nucleus_detection_v0",
    split_positions=True,
    desired_resolution=[1, 1, 1],
)[["id", "pt_position_x", "pt_position_y", "pt_position_z", "pt_root_id", "volume"]]

nuc_df = nuc_df[nuc_df["pt_root_id"] != 0]
print(nuc_df.shape)
nuc_df.head()

### deduplicate cells

In [ ]:
# nucleus_detection_v0 is the raw table: one nucleus can map to several roots and one root to several nuclei.
pairs = nuc_df[["id", "pt_root_id"]].drop_duplicates()
roots_per_nucleus = pairs.groupby("id")["pt_root_id"].nunique()
nuclei_per_root = pairs.groupby("pt_root_id")["id"].nunique()

ambiguous = nuc_df["id"].isin(
    roots_per_nucleus.index[roots_per_nucleus > 1]
) | nuc_df["pt_root_id"].isin(nuclei_per_root.index[nuclei_per_root > 1])

print(f"dropped {int(ambiguous.sum())} / {len(nuc_df)}")
nuc_df = nuc_df[~ambiguous].reset_index(drop=True)

### transform coordinates

`minnie_transform_nm()` accepts nanometers and returns microns. Save its output directly; dividing by 1,000 again would incorrectly store millimetres.

In [ ]:
tform = standard_transform.minnie_transform_nm()
xyz = tform.apply(nuc_df[["pt_position_x", "pt_position_y", "pt_position_z"]].values)

nuc_df[["x_medial-lateral", "y_dorsal-ventral", "z_caudal-rostral"]] = xyz
nuc_df.head()

### save

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
nuc_df.drop(columns="volume").to_parquet(
    OUTPUT_DIR / f"minnie_v{VERSION}_nucleus_positions.parquet"
)

## nucleus volume

### save

In [ ]:
nuc_df[["id", "pt_root_id", "volume"]].to_parquet(
    OUTPUT_DIR / f"minnie_v{VERSION}_nucleus_volume.parquet"
)

## proofread status

### cave query

In [ ]:
proof_df = client.materialize.query_table("proofreading_status_and_strategy")[
    [
        "id",
        "pt_root_id",
        "valid_id",
        "status_axon",
        "status_dendrite",
        "strategy_axon",
        "strategy_dendrite",
    ]
]

# valid_id != pt_root_id means the segment changed after the proofreading assessment.
proof_df["assessment_stale"] = proof_df["valid_id"] != proof_df["pt_root_id"]
print(proof_df.shape)
proof_df.head()

In [ ]:
for ss in proof_df['strategy_dendrite'].unique():
    print(len(proof_df[proof_df['strategy_dendrite'] == ss]))

### save

In [ ]:
proof_df.to_parquet(OUTPUT_DIR / f"minnie_v{VERSION}_proofreading_status.parquet")

## neuron nuclei

### cave query

In [ ]:
# nucleus_ref_neuron_svm is the recommended broad neuron roster
neuron_df = client.materialize.query_table("nucleus_ref_neuron_svm")

# target_id is the nucleus id for this reference table; id is its own row id.
neuron_df = neuron_df[
    ["target_id","pt_root_id", "classification_system", "cell_type"]
]
neuron_df = neuron_df.rename(columns={"target_id": "id"})
neuron_df = neuron_df[
    neuron_df["cell_type"].eq("neuron") & neuron_df["pt_root_id"].ne(0)
].reset_index(drop=True)
print(neuron_df.shape)
neuron_df.head()

### save

In [ ]:
neuron_df.to_parquet(OUTPUT_DIR / f"minnie_v{VERSION}_neuron_nuclei.parquet")

## cell types

### cave query

In [ ]:
ct_df = client.materialize.query_table("cell_type_multifeature_combo")[
    ["id", "pt_root_id", "classification_system", "cell_type"]
]
print(ct_df.shape)
ct_df["cell_type"].value_counts()

In [ ]:
ct_df.head()

### save

In [ ]:
ct_df.to_parquet(OUTPUT_DIR / f"minnie_v{VERSION}_cell_types.parquet")